# Load the libraries

In [103]:
# install.packages("openxlsx")
# install.packages("naniar")
library(readxl)
library(openxlsx)
library(naniar)
library(dplyr)

# Import the dataset

In [104]:
# load the dataset
dataset <- read_xlsx("Dataset.xlsx")

# replace the string value "-" with NA
dataset[dataset=="-"]  <- NA

# remove the rows all data item is NA
dataset <- dataset[rowSums(is.na(dataset)) != ncol(dataset), ]

# remove the columns all data item is NA
dataset <- dataset[,colSums(is.na(dataset))<nrow(dataset)]


New names:
• `Intervention Intervention details` -> `Intervention Intervention
  details...97`
• `Intervention Intervention details` -> `Intervention Intervention
  details...98`
• `ADL after treatment SD` -> `ADL after treatment SD...352`
• `ADL after treatment mean` -> `ADL after treatment mean...353`
• `ADL after treatment N` -> `ADL after treatment N...354`
• `ADL Baseline SD` -> `ADL Baseline SD...362`
• `ADL Baseline mean` -> `ADL Baseline mean...363`
• `ADL Baseline N` -> `ADL Baseline N...364`
• `ADL after treatment mean` -> `ADL after treatment mean...391`
• `ADL after treatment N` -> `ADL after treatment N...392`
• `ADL after treatment SD` -> `ADL after treatment SD...393`
• `ADL Baseline mean` -> `ADL Baseline mean...394`
• `ADL Baseline N` -> `ADL Baseline N...395`
• `ADL Baseline SD` -> `ADL Baseline SD...396`
• `` -> `...5226`


# Data exploration

The dataset contains the summary of all the studies included, which were published before 2019. So the dataset is very large.

In [105]:
# show the dataset (omitted because the dataset is too large)
# dataset

In [106]:
ncol(dataset)
nrow(dataset)

[1] 3705

[1] 733

As mentioned, the large dataset has 3705 columns and 733 rows. We then examined the columns that the dataset contains.

In [107]:
# The code below is able excuable from the console
# The outputs of the commands below are available in the folder as data_rsummary.txt and colnames.txt

# sink("data_rsummary.txt")
# summary(dataset)
# sink(file = NULL)

# sink("colnames.txt")
# df_cols
# sink()

Thus, we will select 'Support for carers' and 'Training and education for carers'. But in case of any missingness, we also explore the studies under the intervention types of 'Structured therapeutic psychosocial interventions', 'Comprehensive care interventions', 'Supportive psychosocial interventions', 'Treatment of co-morbidities or additional risks', 'Unstructured therapeutic interventions' and 'Multicomponent interventions', but there is none related to support or education for caregivers.

In [108]:
# relevant_studies  <- c('Support for carers', 'Training and education for carers')
# relevant_studies <- dataset[dataset['Intervention types'] == c('Support for carers', 'Training and education for carers'), ]['Study Identifier']

## Keyword screening

We note that there are quite a lot of depression/caregiver burden related metrics also in the columns. So we output the colnames to a seperate file named colnames.txt for manual screening of useful columns. According to our preliminary screening, we notice a number of metrics to measure caregiver burden. 

Direct measure of care giver burdern:
- **Hinton 2020** and **Uyar 2019:** Zarit Burden Interview (ZBI)
- **Govindakumari 2020:** Care giver burden, unspecified

Neuropsychiatric symptoms of caregivers:
- **Hinton 2020:** Depression/anxiety symptoms, measured with the Patient Health Questionnaire-4 (PHQ-4)
- **Uyar 2019:** Depression symptoms, measured with Beck Depression Inventory (BDI)
- **Uyar 2019:** Anxiety symptoms, measured with Beck Anxiety Inventory (BAI)
- **Uyar 2019:** Neuropsychiatric symptoms, measured with Neuropsychiatric Inventory-Distress (NPI-D)
- **Ghaffari 2019:** Neuropsychiatric symptoms, measured with General Health Questionnaire (GHQ-28)
- **Zarepour 2020:** Anxiety symtoms, measured with the Spielberger questionnaires
- **Pan 2019:** Depression symptoms, measured with 10-item CES-D
- **Uyar 2019:** Quality of life, measured with Quality of Life Scale SF36 (SF36 mental health)

Behavioural metrics of caregivers:
- **Uyar 2019:** Quality of life, measured with Quality of Life Scale SF36 (SF36 physical health)
- **Pan 2019:** Positive/Negative coping, measured with the Simplified Coping Scale
- **Pan 2019:** Mutuality, measured with 15-item Mutuality Scale

And a number of metrics to measure the patient's symptoms as below.

Cognitive function of patients:
- **Chen 2020:** Montreal Cognitive Assessment Scale (MoCA)

Neuropsychiatric symptoms of patients:
- **Chen 2020** and **Pan 2019:** Mini Mental State Examinination (MMSE)
- **Uyar 2019:** Neuropsychiatric symptoms, measured with Neuropsychiatric Inventory–Severity (NPI-S)

Behavioural metrics of patients:
- **Uyar 2019:** Quality of life, measured with Quality of life in Alzheimer’s Disease (QoL-AD)
- **Chen 2020:** Barthel activities of daily living scale (BADL)
- **Chen 2020:** Behavioral pathology assessment scale of Alzheimer’s disease (BEHAVE-AD)
- **Govindakumari 2020:** Quality of life, unspecified
- **Pan 2019:** Quality of life, measured with the 14-item Activities of Daily Living scale (ADL)

It is possible to directly use keywords to simply create a curated dataset from the subset of the two intervention types. However, I choose to first see what are these variables and what kind of studies used the variables, to make sure that no relavent studies are ignored.

In [109]:
df_cols <- colnames(dataset)

In [110]:
reserved_cols <- head(df_cols,19)
reserved_cols

[1] "Study Identifier"                                                                               
 [2] "Year"                                                                                           
 [3] "Intervention"                                                                                   
 [4] "Intervention types"                                                                             
 [5] "Intervention subgroups"                                                                         
 [6] "Country"                                                                                        
 [7] "Setting"                                                                                        
 [8] "Types of disease_clean"                                                                         
 [9] "Type of disease in two categories (dementia and MCI)"                                           
[10] "Updated Dementia Severity"                                                                      
[11] "Is this a study focusing on carers of people living with dementia or mild cognitive impairment?"
[12] "Primary Cognition Measure standardized z-score change"                                          
[13] "Type"                                                                                           
[14] "Baseline Score - Primary Cognition"                                                             
[15] "Number of Participants at Baseline"                                                             
[16] "Baseline SD"                                                                                    
[17] "Post Treatment Score"                                                                           
[18] "Post Treatment N"                                                                               
[19] "Post Treatment SD"

In [111]:
# keywords for direct measurement of caregiver burden
caregiver_burden_keys <- unique(c(
    grep("burden", df_cols),
    grep("Burden", df_cols),
    grep("ZBI", df_cols),
    grep("Zarit", df_cols)
    ))
caregiver_burden_keys <- df_cols[caregiver_burden_keys]
# caregiver_burden_keys 

In [112]:
df_caregiver_burden <- dataset[which(rowSums(is.na(dataset[caregiver_burden_keys]))!=ncol(dataset[caregiver_burden_keys])), ]
df_caregiver_burden <- df_caregiver_burden[rowSums(is.na(df_caregiver_burden)) != ncol(df_caregiver_burden), ]

# df_caregiver_burden <- df_caregiver_burden[c(df_caregiver_burden, caregiver_burden_keys)]
df_caregiver_burden <- df_caregiver_burden[,colSums(is.na(df_caregiver_burden))<nrow(df_caregiver_burden)]
dim(df_caregiver_burden)

[1]  31 286

In [113]:
# df_caregiver_burden <- df_caregiver_burden[,df_caregiver_burden['Intervention types'] %in% as.list(relevant_studies)]
df_caregiver_burden <- df_caregiver_burden[,colSums(is.na(df_caregiver_burden))<nrow(df_caregiver_burden)]
dim(df_caregiver_burden)
# write.csv(df_caregiver_burden, 'Dataset_CaregiverBurden_all.csv', row.names=FALSE)

[1]  31 286

Then, the dataset with all caregiver burden data will be manually curated. Similarly, we perform the same filtering strategy to other measurements.

In [114]:
# df_caregiver_burden[df_caregiver_burden['Study Identifier'] %in% relevant_studies]

Neuropsychiatric symptoms of caregivers:
- **Hinton 2020:** Depression/anxiety symptoms, measured with the Patient Health Questionnaire-4 (PHQ-4)
- **Uyar 2019:** Depression symptoms, measured with Beck Depression Inventory (BDI)
- **Uyar 2019:** Anxiety symptoms, measured with Beck Anxiety Inventory (BAI)
- **Uyar 2019:** Neuropsychiatric symptoms, measured with Neuropsychiatric Inventory-Distress (NPI-D)
- **Ghaffari 2019:** Neuropsychiatric symptoms, measured with General Health Questionnaire (GHQ-28)
- **Zarepour 2020:** Anxiety symtoms, measured with the Spielberger questionnaires
- **Pan 2019:** Depression symptoms, measured with 10-item CES-D
- **Uyar 2019:** Quality of life, measured with Quality of Life Scale SF36 (SF36 mental health)

Neuropsychiatric symptoms of patients:
- **Chen 2020** and **Pan 2019:** Mini Mental State Examinination (MMSE)
- **Uyar 2019:** Neuropsychiatric symptoms, measured with Neuropsychiatric Inventory–Severity (NPI-S)

In [115]:
# keywords for neuropsychiatric symptoms:
mental_health_keys <- unique(c(

    # abbreviations
    grep("PHQ", df_cols),
    grep("SF36", df_cols),
    grep("NPI", df_cols),
    grep("GHQ", df_cols),
    grep("PHQ", df_cols),
    grep("BDI", df_cols),
    grep("BAI", df_cols),
    # grep("MMSE", df_cols), # we don't include as this is mostly for care recipients

    # words and phrases
    grep("Neuropsychiatric", df_cols),
    grep("neuropsychiatric", df_cols), 
    grep("Depression", df_cols),
    grep("depression", df_cols),
    grep("Anxiety", df_cols),
    grep("anxiety", df_cols)
    ))

# select mental health keys and drop columns and rows with only NAs
mental_health_keys <- df_cols[mental_health_keys]

In [116]:
df_mental_health <- dataset[which(rowSums(is.na(dataset[mental_health_keys]))!=ncol(dataset[mental_health_keys])), ]
# df_mental_health <- df_mental_health[, c(reserved_cols, mental_health_keys)]
# df_mental_health <- df_mental_health[df_mental_health$'Study Identifier' == intervention_types_reserved]
df_mental_health <- df_mental_health[rowSums(is.na(df_mental_health)) != ncol(df_mental_health), ]
df_mental_health <- df_mental_health[,colSums(is.na(df_mental_health))<nrow(df_mental_health)]

# print dimensions of the dataset
dim(df_mental_health)

[1]  40 816

In [117]:
# select intervention types
df_mental_health <- df_mental_health[,colSums(is.na(df_mental_health))<nrow(df_mental_health)]

# print dimensions of the dataset
dim(df_mental_health)

# write the uncurated data file
# write.csv(df_mental_health, 'Dataset_MentalHealth_all.csv', row.names=FALSE)

[1]  40 816

Behavioural metrics of caregivers:
- **Uyar 2019:** Quality of life, measured with Quality of Life Scale SF36 (SF36 physical health)
- **Pan 2019:** Positive/Negative coping, measured with the Simplified Coping Scale
- **Pan 2019:** Mutuality, measured with 15-item Mutuality Scale

Behavioural metrics of patients:
- **Uyar 2019:** Quality of life, measured with Quality of life in Alzheimer’s Disease (QoL-AD)
- **Chen 2020:** Barthel activities of daily living scale (BADL)
- **Chen 2020:** Behavioral pathology assessment scale of Alzheimer’s disease (BEHAVE-AD)
- **Govindakumari 2020:** Quality of life, unspecified
- **Pan 2019:** Quality of life, measured with the 14-item Activities of Daily Living scale (ADL)

In [118]:
# keywords for behavioural metrics:
behvaioural_keys <- unique(c(

    # abbreviations
    grep("SF36", df_cols),
    grep("QoL", df_cols),
    grep("BADL", df_cols),
    grep("BEHAVE", df_cols),
    grep("ADL", df_cols),

    # words and phrases
    grep("Quality", df_cols),
    grep("quality", df_cols), 
    grep("Coping", df_cols),
    grep("coping", df_cols),
    grep("Mutuality", df_cols),
    grep("mutuality", df_cols),
    grep("daily", df_cols),
    grep("Daily", df_cols),
    grep("Barthel", df_cols),
    grep("Behaviour", df_cols),
    grep("Behaviour", df_cols),
    grep("behavior", df_cols),
    grep("behavior", df_cols)
    ))

# select behavioural keys and drop columns and rows with only NAs
behvaioural_keys <- df_cols[behvaioural_keys]
# behvaioural_keys[-grepl('sleep', behvaioural_keys)]

In [119]:
df_behavioural <- dataset[which(rowSums(is.na(dataset[behvaioural_keys]))!=ncol(dataset[behvaioural_keys])), ]
# df_behavioural <- df_behavioural[, c(reserved_cols, behvaioural_keys)]
df_behavioural <- df_behavioural[rowSums(is.na(df_behavioural)) != ncol(df_behavioural), ]
df_behavioural <- df_behavioural[,colSums(is.na(df_behavioural))<nrow(df_behavioural)]

# print dimensions of the dataset
dim(df_behavioural)

[1]  78 933

In [120]:
# select intervention types
# df_behavioural <- df_behavioural[df_behavioural['Intervention types'] == intervention_types_reserved, ]
df_behavioural <- df_behavioural[,colSums(is.na(df_behavioural))<nrow(df_behavioural)]
df_behavioural <- df_behavioural[rowSums(is.na(df_behavioural)) != ncol(df_behavioural), ]

# print dimensions of the dataset
dim(df_behavioural)
# df_behavioural

# write the uncurated data file
# write.csv(df_behavioural, 'Dataset_Behavioural_all.csv', row.names=FALSE)

[1]  78 933

In [121]:
# write.xlsx(df_behavioural, "Dataset_data_filtered.xlsx", sheetName = "caregiver_Behaviour")
# write.xlsx(df_mental_health, "Dataset_data_filtered.xlsx", sheetName = "caregiver_MentalHealth", append = TRUE)
# write.xlsx(df_caregiver_burden, "Dataset_data_filtered.xlsx", sheetName = "caregiver_Burden", append = TRUE)

# Intervention detail screening

Here we aim to filter the studies by the intervention type. First we can have a look at the column names related to  "Intervention"

In [122]:
df_cols[grep('Intervention', df_cols)]

[1] "Intervention"                                                                                                                                                                                                                                                   
 [2] "Intervention types"                                                                                                                                                                                                                                             
 [3] "Intervention subgroups"                                                                                                                                                                                                                                         
 [4] "Intervention Brief description of intervention"                                                                                                                                                                                                                 
 [5] "Intervention Brief description ofi ntervention"                                                                                                                                                                                                                 
 [6] "Intervention Brief intervention description"                                                                                                                                                                                                                    
 [7] "Intervention Brief introduction to the intervention"                                                                                                                                                                                                            
 [8] "Intervention Intervention details...97"                                                                                                                                                                                                                         
 [9] "Intervention Intervention details...98"                                                                                                                                                                                                                         
[10] "Intervention Intervention details (duration and intensity of the intervention, dosage of drugs, existence of a protocol or manual for psychosocial, training or education interventions)"                                                                       
[11] "Intervention intervention type"                                                                                                                                                                                                                                 
[12] "Intervention Intervention type.1"                                                                                                                                                                                                                               
[13] "Intervention Interv. Type:"                                                                                                                                                                                                                                     
[14] "Intervention Interv. type [SELECT: Pharmaceutical; Traditional Chinese Medicine; Other trad. medicines; Non-pharmacol.; Multicomponent (record each component); Interv. for carers; End of life care; Treatment/prevention of co-morbidities; Technology; Diagn"
[15] "Intervention Interv. type [SELECT: Pharmaceutical; Traditional Chinese Medicine; Other trad. medicines; Non-pharmacol.; Multicomponent (record each component); Interv. for carers; End of life care; Treatment/preventioo-morbidities; Technology; Diagnostic;"
[16] "Intervention Name of intervention"                

In [123]:
as.data.frame(table(dataset['Intervention types']))

Intervention.types,Freq
<fct>,<int>
Comprehensive care interventions,7
Electrical brain stimulation,14
Multicomponent interventions,28
Music interventions,8
No intervention,251
Nutrition,4
Other interventions,2
Pharmaceutical,130
Physical exercise,17


We can safely exclude "Pharmaceutical", "Traditional Chinese Medicine", "Supplements", "Physical exercise", "Nutrition", "Electrical brain stimulation", "Treatment of co-morbidities or additional risks", "Music interventions". 

In [124]:
df_studies <- dataset[dataset$'Intervention types' %in% c(
    "Structured therapeutic psychosocial interventions", 
    "Training and education for carers", 
    "Other interventions", 
    "Support for carers",
    "Comprehensive care interventions",
    "Supportive psychosocial interventons",
    "Treatment of co-morbidities or additional risks", 
    "Unstructured therapeutic interventions",
    "Multicomponent interventions"), ]
dim(df_studies)

[1]  101 3705

Thus, we reduce the scope of the analysis from 733 studies to only 101 of them. Then we screen the further details of the intervention.

In [125]:
# as.data.frame(table(df_studies$"Is this a study focusing on carers of people living with dementia or mild cognitive impairment?"))
as.data.frame(table(df_studies$'Intervention subgroups'))

Var1,Freq
<fct>,<int>
Cognitive stimulation therapy,2
Cognitive training,14
Combination of structured psychosocial interventions,3
Community-based interventions,1
Group support and therapy,7
Individual-based interventions,6
"Information, education, and training to support carers",7
Multicomponent TCM,7
Multicomponent TCM + psychosocial or nursing intervention,4


We surely do not want to include any drug-related intervention, so we exclude the terms including "TCM", "pharmaceutical", "nutrition", "therapeutic substance".

In [126]:
included_terms <- unique(df_studies$'Intervention subgroups')[- grep("TCM|pharmaceutical|nutrition|therapeutic substance", unique(df_studies$'Intervention subgroups'))]
df_studies <- df_studies[df_studies$'Intervention subgroups' %in% included_terms, ]
df_studies <- df_studies[,colSums(is.na(df_studies))<nrow(df_studies)]
dim(df_studies)

[1]   79 1244

In [127]:
# unique(df_studies['Intervention subgroups'])
# unique(df_studies['Intervention Brief description of intervention'])
# ...

In [128]:
write.xlsx(df_studies, "Dataset_intervention_screening.xlsx")

Then based on the descriptions in the columns "Intervention", "Intervention Brief description of intervention", "Intervention Intervention details...97", we mannually generate a list of eligible studies. The selected studies are listed in selected_studies.txt. 

The selection is based on whether there is caregiver support and education in the detailed description of the study. However, there is one study I think does not have enough detail for the intervention. Lin 2015 only describes the intervention names SGGI and LGGI, but doesn't provide any details about the intervention.

In [129]:
# df_studies[c(reserved_cols, caregiver_burden_keys)]
# df_behavioural <- df_behavioural[,colSums(is.na(df_behavioural))<nrow(df_behavioural)]
# df_behavioural <- df_behavioural[rowSums(is.na(df_behavioural)) != ncol(df_behavioural), ]

In [130]:
selected_studies <- read.csv('selected_studies.txt')
selected_studies <- unique(selected_studies$Study.Identifier)
length(selected_studies)

[1] 27

In [131]:
selected_studies

[1] "Wang 2010b"            "ZHAO 2010"             "SUN 2010"             
 [4] "SU 2012"               "JIANG 2012"            "Wang 2014a"           
 [7] "Yang 2017a"            "He 2012"               "Novelli 2018"         
[10] "Lok 2019"              "Pahlavanzadeh 2010"    "Wang 2012e"           
[13] "Aboulafia-Brakha 2014" "Mercedeh 2014"         "Kamkhagi 2015"        
[16] "Söylemez 2016"         "Shata 2017"            "Salamizadeh 2017"     
[19] "Mahdavi 2017"          "Lök 2017"              "Orawan 2018"          
[22] "Amit 2008"             "TAN 2010"              "Guerra 2011"          
[25] "Wang 2017e"            "Wang 2017c"            "Lin 2017"

In [132]:
df_caregiver_burden <- df_caregiver_burden[df_caregiver_burden$'Study Identifier' %in% as.list(selected_studies), ]
# df_caregiver_burden <- df_caregiver_burden[c(reserved_cols, caregiver_burden_keys), ]
df_caregiver_burden <- df_caregiver_burden[,colSums(is.na(df_caregiver_burden))<nrow(df_caregiver_burden)]

df_mental_health <- df_mental_health[df_mental_health$'Study Identifier' %in% as.list(selected_studies), ]
# df_mental_health <- df_mental_health[c(reserved_cols, mental_health_keys), ]
df_mental_health <- df_mental_health[,colSums(is.na(df_mental_health))<nrow(df_mental_health)]

df_behavioural <- df_behavioural[df_behavioural$'Study Identifier' %in% as.list(selected_studies), ]
# df_behavioural <- df_behavioural[c(reserved_cols, behvaioural_keys), ]
df_behavioural <- df_behavioural[,colSums(is.na(df_behavioural))<nrow(df_behavioural)]

df_caregiver_burden

In [133]:
dim(df_caregiver_burden)
dim(df_mental_health)
dim(df_behavioural)

[1]  22 164

[1]  4 87

[1]  16 163

In [134]:
wb <- createWorkbook()

addWorksheet(wb, "caregiver_Burden")
addWorksheet(wb, "caregiver_MentalHealth")
addWorksheet(wb, "caregiver_Behaviour")

writeData(wb, "caregiver_Burden", df_caregiver_burden)
writeData(wb, "caregiver_MentalHealth", df_mental_health)
writeData(wb, "caregiver_Behaviour", df_behavioural)

saveWorkbook(wb, "Dataset_data_filtered_final.xlsx", overwrite = TRUE)

#write.xlsx(df_caregiver_burden, "Dataset_data_filtered_final.xlsx", sheetName = "caregiver_Burden")
#write.xlsx(df_mental_health, "Dataset_data_filtered_final.xlsx", sheetName = "caregiver_MentalHealth", append = TRUE)
#write.xlsx(df_behavioural, "Dataset_data_filtered_final.xlsx", sheetName = "caregiver_Behaviour", append = TRUE)